# Old metrics vs the sendable gate, on real examples
This notebook puts the **old evaluation** (averages of toxicity, similarity and fact recall) next to the **new sendable gate** (`src/secondthought/sendable_gate.py`, proposed in `data/workplace_eval/new_proposed_eval.md`) on the same 35 workplace messages.

- **Old:** each metric is reported separately and averaged. A rewrite can look good on every number and still be one you would not send.
- **New:** each rewrite gets one verdict, *sendable or not*, and the checks it failed.

The examples are grouped by how hostile the **input** is: mild, medium and severe.

It only reads the scored CSVs in `outputs/`, so no models are downloaded or run.

**The gate is a prototype.** Two of its six checks are not built yet (intent kept; nothing added and fluent), and its thresholds are placeholders until they are fitted to human labels. Its sendable rates are therefore an upper bound.

## 1. Load the three systems and run the gate

In [1]:
import csv
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from secondthought.sendable_gate import CHECKS, gate, wilson

# All three ran on the same 35 messages. The GPT baseline used a different 10-message set, so it is left out.
SYSTEMS = {
    "ParaDetox T5": "outputs/epoch3_evaluated_updated.csv",
    "both-t5 (detox → formal)": "outputs/both_detox_formalize.csv",
    "Chained single-task T5s": "outputs/t5_chain_evaluated.csv",
}


def load(path):
    rows = list(csv.DictReader(open(ROOT / path, encoding="utf-8")))
    return pd.DataFrame([{**row, **gate(row)} for row in rows]).set_index("id")


frames = {name: load(path) for name, path in SYSTEMS.items()}
for name, frame in frames.items():
    note = "" if "toxicity_rewritten" in frame else "  (no Detoxify scores: hostility uses the word list only)"
    print(f"{name}: {len(frame)} rewrites{note}")

ParaDetox T5: 35 rewrites
both-t5 (detox → formal): 35 rewrites
Chained single-task T5s: 35 rewrites  (no Detoxify scores: hostility uses the word list only)


## 2. Intensity of the input
The intensity of each message is Detoxify's toxicity score on the **original** text: **mild** below 0.3, **medium** 0.3–0.8, **severe** 0.8 and above.

Some "mild" messages are clearly rude to a person ("You've screwed up the address again" scores 0.002). That is the first sign that Detoxify alone misses workplace hostility.

In [2]:
input_toxicity = pd.to_numeric(frames["ParaDetox T5"]["toxicity_original"])
original_text = frames["ParaDetox T5"]["original_text"]


def intensity(score):
    return "mild" if score < 0.3 else "medium" if score < 0.8 else "severe"


level = input_toxicity.map(intensity)
LEVELS = ["mild", "medium", "severe"]
display(level.value_counts().reindex(LEVELS).rename("messages").to_frame())
display(pd.DataFrame({"input toxicity": input_toxicity.round(3), "level": level, "message": original_text})
        .sort_values("input toxicity").groupby("level", sort=False).head(3))

,messages
toxicity_original,
mild,9
medium,12
severe,14


,input toxicity,level,message
id,,,
input_012,0.002,mild,You've screwed up the address again. It's 160 ...
input_021,0.005,mild,Quit screwing around with the schedule. Luis s...
input_023,0.022,mild,"Rachel, your budget is a joke. We approved $60..."
input_020,0.460,medium,"You left the customer waiting for 45 minutes, ..."
input_033,0.471,medium,"This design is ugly as hell, and the tiny labe..."
input_017,0.473,medium,Your careless packing is ridiculous. Put 2 kg ...
input_026,0.806,severe,"I can't believe this crap, Aisha. You promised..."
input_031,0.825,severe,I'm tired of your lazy half-assed answers. Exp...
input_005,0.859,severe,This lazy crap has cost us 2 hours already. Re...


## 3. The old view vs the new view, per system
The old columns are what we reported in earlier sessions. The last column is the gate's verdict, with a 95% confidence interval.

In [4]:
def number(frame, column):
    return pd.to_numeric(frame[column], errors="coerce") if column in frame else None


def true_rate(frame, column):
    return frame[column].astype(str).str.lower().eq("true").mean()


rows = []
for name, frame in frames.items():
    toxicity, reduction, similarity = (number(frame, c) for c in
        ("toxicity_rewritten", "toxicity_relative_reduction", "semantic_similarity"))
    sendable = int(frame["gate_sendable"].sum())
    low, high = wilson(sendable, len(frame))
    rows.append({
        "System": name,
        "OLD: toxicity after": "—" if toxicity is None else f"{toxicity.mean():.3f}",
        "OLD: relative reduction": "—" if reduction is None else f"{reduction.mean():.0%}",
        "OLD: similarity": "—" if similarity is None else f"{similarity.mean():.3f}",
        "OLD: facts exact": f"{true_rate(frame, 'critical_value_exact'):.0%}",
        "OLD: exact copies": int(frame["exact_match"].astype(str).str.lower().eq("true").sum()),
        "NEW: sendable": f"{sendable}/{len(frame)} ({sendable / len(frame):.0%}, CI {low:.0%}–{high:.0%})",
    })
pd.DataFrame(rows).set_index("System")

,OLD: toxicity after,OLD: relative reduction,OLD: similarity,OLD: facts exact,OLD: exact copies,NEW: sendable
System,,,,,,
ParaDetox T5,0.056,84%,0.967,100%,4,"25/35 (71%, CI 55%–84%)"
both-t5 (detox → formal),0.060,81%,0.955,94%,2,"24/35 (69%, CI 52%–81%)"
Chained single-task T5s,—,—,—,91%,2,"25/35 (71%, CI 55%–84%)"


**Reading it:** on the old numbers, the ParaDetox T5 looks best: lowest toxicity after rewriting, biggest reduction, highest similarity, and every fact kept. The gate rates all three at roughly 70% sendable, with overlapping intervals, so on 35 messages there is no evidence that one is better than another. The old ranking came from averages that hide which messages are still unsendable.

## 4. Why rewrites fail: failures per check
A rewrite can fail several checks, and every failure is counted. "not measured" means the check is not built yet, or this file lacks the data it needs.

In [5]:


failures = {}
for name, frame in frames.items():
    column = {}
    for check in CHECKS:
        values = frame[f"gate_{check}"]
        column[check] = "not measured" if values.isna().all() else int((values == False).sum())  # noqa: E712
    failures[name] = column
pd.DataFrame(failures)

,ParaDetox T5,both-t5 (detox → formal),Chained single-task T5s
control_unchanged,not measured,not measured,not measured
not_copy,4,2,2
facts_intact,0,2,3
hostility_gone,7,8,7
intent_kept,not measured,not measured,not measured
nothing_lost,3,1,0
nothing_added,not measured,not measured,not measured


## 5. Sendable rate by input intensity

In [6]:
by_level = pd.DataFrame({
    name: frame["gate_sendable"].groupby(level).agg(lambda s: f"{int(s.sum())}/{len(s)}").reindex(LEVELS)
    for name, frame in frames.items()
})
by_level

,ParaDetox T5,both-t5 (detox → formal),Chained single-task T5s
toxicity_original,,,
mild,7/9,5/9,6/9
medium,7/12,7/12,8/12
severe,11/14,12/14,11/14


Severe inputs pass most often, because their insults are obvious swear words that every system removes. Mild and medium inputs do worse: their hostility is in words like "is a joke", "screwed up" and "ridiculous", which Detoxify scores low. The models therefore leave those words in, or return the message unchanged, and the gate fails them.

## 6. One real example per intensity level
For each example: the input, each system's rewrite, the **old** metrics, and the **new** verdict with the failed checks.

Change `FEATURED` to look at any other message (ids are `input_001` to `input_035`).

In [7]:
FEATURED = {"mild": "input_023", "medium": "input_017", "severe": "input_032"}


def fmt(value, pattern):
    try:
        return pattern.format(float(value))
    except (TypeError, ValueError):
        return "—"


def show_example(message_id):
    display(Markdown(
        f"### {level[message_id].upper()} · `{message_id}` · Detoxify on input: {input_toxicity[message_id]:.3f}\n\n"
        f"> {original_text[message_id]}"
    ))
    table = []
    for name, frame in frames.items():
        row = frame.loc[message_id]
        table.append({
            "System": name,
            "Rewrite": row["rewritten_text"],
            "OLD: toxicity after": fmt(row.get("toxicity_rewritten"), "{:.3f}"),
            "OLD: reduction": fmt(row.get("toxicity_relative_reduction"), "{:.0%}"),
            "OLD: similarity": fmt(row.get("semantic_similarity"), "{:.3f}"),
            "OLD: facts exact": row["critical_value_exact"],
            "NEW: verdict": "✅ sendable" if row["gate_sendable"] else "❌ not sendable",
            "NEW: failed checks": row["gate_all_failures"].replace(";", ", ") or "—",
            "NEW: harsh words left": row["gate_hostile_markers"].replace(";", ", ") or "—",
        })
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(table).set_index("System"))

In [8]:
show_example(FEATURED["mild"])

### MILD · `input_023` · Detoxify on input: 0.022

> Rachel, your budget is a joke. We approved $600 for travel, not $900.

,Rewrite,OLD: toxicity after,OLD: reduction,OLD: similarity,OLD: facts exact,NEW: verdict,NEW: failed checks,NEW: harsh words left
System,,,,,,,,
ParaDetox T5,"Rachel, your budget is a joke. We approved $600 for travel, not $900.",0.022,0%,1.000,True,❌ not sendable,"not_copy, hostility_gone",is a joke
both-t5 (detox → formal),"Rachel, your budget is a joke. We approved $600 for travel, not $900.",0.022,0%,1.000,True,❌ not sendable,"not_copy, hostility_gone",is a joke
Chained single-task T5s,"Rachel, your budget is a joke. We approved $600 for travel, not $900.",—,—,—,True,❌ not sendable,"not_copy, hostility_gone",is a joke


**Mild:** every system returned the message **unchanged**. The old metrics rate that as perfect: similarity 1.0, every fact kept, and near-zero toxicity, because Detoxify never saw the input as toxic. The gate fails it as a copy that still contains "is a joke".

In [9]:
show_example(FEATURED["medium"])

### MEDIUM · `input_017` · Detoxify on input: 0.473

> Your careless packing is ridiculous. Put 2 kg of ice in each cooler, not 200 g.

,Rewrite,OLD: toxicity after,OLD: reduction,OLD: similarity,OLD: facts exact,NEW: verdict,NEW: failed checks,NEW: harsh words left
System,,,,,,,,
ParaDetox T5,"Your careless packing is ridiculous. Put 2 kg of ice in each cooler, not 200 g.",0.473,0%,1.000,True,❌ not sendable,"not_copy, hostility_gone",ridiculous
both-t5 (detox → formal),"Your packing is ridiculous. Put 2 kg of ice in each cooler, not 200 g.",0.516,-9%,0.985,True,❌ not sendable,hostility_gone,ridiculous
Chained single-task T5s,"Your careless packing is not good. Put 2 kg of ice in each cooler, not 200 g.",—,—,—,True,✅ sendable,—,—


**Medium:** the systems behave differently. Two keep "ridiculous" and fail the gate. Here the old toxicity score agrees, showing 0% and −9% reduction. The chained models replace it with "not good" and pass. The old view cannot judge that file at all, because it has no Detoxify scores. The gate still can, because its harsh-word check needs no model.

In [10]:
show_example(FEATURED["severe"])

### SEVERE · `input_032` · Detoxify on input: 0.986

> You're being a complete asshole about this. Let the new hire finish speaking.

,Rewrite,OLD: toxicity after,OLD: reduction,OLD: similarity,OLD: facts exact,NEW: verdict,NEW: failed checks,NEW: harsh words left
System,,,,,,,,
ParaDetox T5,Let the new hire finish speaking.,0.001,100%,0.784,True,❌ not sendable,nothing_lost,—
both-t5 (detox → formal),You are being unsuitable about this. Let the new hire finish speaking.,0.001,100%,0.770,True,✅ sendable,—,—
Chained single-task T5s,You are being unsuitable about this. Let the new hire finish speaking.,—,—,—,True,✅ sendable,—,—


**Severe:** the ParaDetox model **deleted the whole first sentence**. Under the old metrics that looks excellent: toxicity near zero, all facts kept, similarity still 0.78. The gate fails it because the criticism disappeared instead of being rephrased. The other two keep the point ("You are being unsuitable about this") and pass, although "unsuitable" is a clumsy word that the planned fluency check should judge.

## 7. Where the old and new evaluations disagree
Here "**Old says good**" means toxicity fell by at least half, every fact was kept, and similarity is at least 0.9: the bar a rewrite would have cleared in earlier reports. The chained-models file has no toxicity or similarity scores, so it is left out of this comparison.

In [12]:
for name, frame in frames.items():
    if "toxicity_relative_reduction" not in frame:
        continue
    old_good = (
        (pd.to_numeric(frame["toxicity_relative_reduction"]) >= 0.5)
        & frame["critical_value_exact"].astype(str).str.lower().eq("true")
        & (pd.to_numeric(frame["semantic_similarity"]) >= 0.9)
    )
    new_good = frame["gate_sendable"].astype(bool)
    display(Markdown(f"**{name}:** old good & new good {int((old_good & new_good).sum())} · "
                     f"old good but gate fails **{int((old_good & ~new_good).sum())}** · "
                     f"old bad but gate passes **{int((~old_good & new_good).sum())}** · "
                     f"both bad {int((~old_good & ~new_good).sum())}"))
    with pd.option_context("display.max_colwidth", None):
        display(frame.loc[old_good & ~new_good, ["original_text", "rewritten_text", "gate_all_failures"]].head(5))

**ParaDetox T5:** old good & new good 25 · old good but gate fails **2** · old bad but gate passes **0** · both bad 8

,original_text,rewritten_text,gate_all_failures
id,,,
input_001,"Seriously, Maya, this invoice is a damn mess. Change the total from $480 to $450 before you send it.","Seriously, Maya, this invoice is a mess. Change the total from $480 to $450 before you send it.",hostility_gone
input_028,Your handover notes are useless garbage. Tell me what's broken and who's supposed to fix it.,Your handover notes are useless. Tell me what's broken and who's supposed to fix it.,hostility_gone


**both-t5 (detox → formal):** old good & new good 22 · old good but gate fails **2** · old bad but gate passes **2** · both bad 9

,original_text,rewritten_text,gate_all_failures
id,,,
input_001,"Seriously, Maya, this invoice is a damn mess. Change the total from $480 to $450 before you send it.","Seriously, Maya, this invoice is a mess. Change the total from $480 to $450 before you send it.",hostility_gone
input_028,Your handover notes are useless garbage. Tell me what's broken and who's supposed to fix it.,Your handover notes are useless. Tell me what is broken and who is supposed to fix it.,hostility_gone


- **Old good, gate fails:** rewrites that score well on every number but still contain a harsh word Detoxify does not flag ("useless", "screwed up", "ridiculous"), or that dropped part of the message.
- **Old bad, gate passes:** the insult was replaced rather than deleted ("complete asshole" became "unsuitable"), so similarity fell just below 0.9. That is the rephrasing we want, but the old similarity bar punishes it. A similarity threshold alone cannot tell rephrasing an insult apart from changing the meaning.

## 8. Known gaps: failures the gate still misses
These should start failing once the remaining checks are built. Until then the gate passes them.

In [13]:
show_example("input_003")

### SEVERE · `input_003` · Detoxify on input: 0.940

> I'm sick of your bullshit excuses, Daniel. The client needs those 6 drawings by Friday.

,Rewrite,OLD: toxicity after,OLD: reduction,OLD: similarity,OLD: facts exact,NEW: verdict,NEW: failed checks,NEW: harsh words left
System,,,,,,,,
ParaDetox T5,I'm sick of your excuses Daniel. The client needs those 6 drawings by Friday.,0.069,93%,0.982,True,✅ sendable,—,—
both-t5 (detox → formal),I am sick of your excuses Daniel. The client needs those six drawings by Friday.,0.059,94%,0.978,True,✅ sendable,—,—
Chained single-task T5s,I am sick of your excuses Daniel. The client needs those six drawings by Friday.,—,—,—,True,✅ sendable,—,—


- "I am sick of your excuses" is still hostile, but no listed word is left and Detoxify scores it low. The planned **workplace hostility score** (check 3) should catch it.
- "those **six** drawings" is fine: the fact checker treats "6" and "six" as the same value.

## 9. Caveats
- **Provisional:** *intent kept* and *nothing added / fluent* are not built yet, so the real sendable rates will be lower.
- **Thresholds are placeholders** (Detoxify 0.2, length ratio 0.75) until they are fitted to human answers to "Would you send this to your manager as-is?".
- **The harsh-word list comes from `extension/triggers.js`,** which was written while looking at these same 35 messages, so the gate is flattered on this set.
- **35 messages is a development set,** and the intervals are wide. A locked test set of new messages is needed before any system is declared better.
- **Some calls are debatable** (is "this invoice is a mess" still hostile?). That is exactly what the human labels should settle.